# Analisis Association Rule pada Dataset Online Retail

**Tujuan:** menemukan kombinasi produk yang sering dibeli bersama dan menyusun rekomendasi cross-selling.

**Metode:** FP-Growth dengan pustaka `mlxtend`, dilanjutkan dengan pembentukan *association rules*.

**Ukuran evaluasi:**
- **Support**: proporsi invoice yang memuat seluruh item dalam aturan.
- **Confidence**: peluang consequent muncul jika antecedent sudah ada.
- **Lift**: kekuatan asosiasi dibandingkan kemunculan acak. Nilai `lift > 1` menunjukkan asosiasi positif.

## 1. Import dan Pemahaman Dataset

Dataset berisi transaksi ritel dari sebuah toko online. Setiap baris merupakan satu baris item dalam invoice, sehingga satu invoice dapat terdiri atas beberapa baris.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 60)

In [2]:
df = pd.read_excel("Online Retail.xlsx")

print(f"Jumlah baris : {df.shape[0]:,}")
print(f"Jumlah kolom : {df.shape[1]}")
print(f"Shape        : {df.shape}")

display(df.head())
df.info()

Jumlah baris : 541,909
Jumlah kolom : 8
Shape        : (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB


**Struktur data:** kolom `InvoiceNo` dan `Description` digunakan untuk membentuk keranjang belanja. Kolom `Quantity` dipakai untuk menyaring pembatalan/retur, sedangkan `InvoiceDate`, `UnitPrice`, `CustomerID`, dan `Country` tidak diperlukan untuk perhitungan aturan asosiasi.

Perhatikan bahwa `CustomerID` memiliki banyak nilai kosong, tetapi kolom ini tidak dihapus karena analisis dilakukan pada level invoice, bukan level pelanggan.

In [3]:
missing = pd.DataFrame({
    "Jumlah Missing": df.isna().sum(),
    "Persentase (%)": (df.isna().mean() * 100).round(2)
}).sort_values("Jumlah Missing", ascending=False)

display(missing)

rows_before = len(df)
invoices_before = df["InvoiceNo"].nunique()

print(f"Jumlah baris sebelum pembersihan   : {rows_before:,}")
print(f"Jumlah invoice sebelum pembersihan : {invoices_before:,}")
print(f"Periode transaksi                  : {df['InvoiceDate'].min()} s.d. {df['InvoiceDate'].max()}")

,Jumlah Missing,Persentase (%)
CustomerID,135080,24.93
Description,1454,0.27
StockCode,0,0.00
InvoiceNo,0,0.00
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
Country,0,0.00


Jumlah baris sebelum pembersihan   : 541,909
Jumlah invoice sebelum pembersihan : 25,900
Periode transaksi                  : 2010-12-01 08:26:00 s.d. 2011-12-09 12:50:00


## 2. Eksplorasi dan Pembersihan Data

Pembersihan mengikuti ketentuan praktikum:
1. menghapus baris dengan `Description` kosong,
2. menghapus invoice yang dibatalkan (`InvoiceNo` diawali huruf `C`),
3. menghapus baris dengan `Quantity` negatif atau nol.

Spasi berlebih pada deskripsi juga dinormalisasi agar produk dengan nama yang sama tidak terpecah menjadi beberapa item.

In [4]:
df_clean = df.copy()

# Normalisasi spasi pada nama produk
df_clean["Description"] = (
    df_clean["Description"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

mask_missing = df_clean["Description"].isna() | df_clean["Description"].fillna("").eq("")
mask_cancel = df_clean["InvoiceNo"].astype(str).str.startswith("C")
mask_quantity = df_clean["Quantity"] <= 0
mask_total = mask_missing | mask_cancel | mask_quantity

filter_summary = pd.DataFrame({
    "Kriteria": [
        "Description kosong",
        "Invoice dibatalkan (InvoiceNo diawali C)",
        "Quantity <= 0",
        "Total baris yang dihapus (gabungan)"
    ],
    "Jumlah Baris": [
        int(mask_missing.sum()),
        int(mask_cancel.sum()),
        int(mask_quantity.sum()),
        int(mask_total.sum())
    ]
})

display(filter_summary)

df_clean = df_clean.loc[~mask_total].copy()

rows_after = len(df_clean)
invoices_after = df_clean["InvoiceNo"].nunique()

comparison = pd.DataFrame({
    "Metrik": ["Jumlah baris", "Jumlah invoice unik"],
    "Sebelum": [rows_before, invoices_before],
    "Sesudah": [rows_after, invoices_after]
})
comparison["Dihapus"] = comparison["Sebelum"] - comparison["Sesudah"]

display(comparison)

print("Catatan: jumlah pada masing-masing kriteria dapat tumpang tindih, sehingga total gabungan tidak sama dengan penjumlahan sederhana.")

,Kriteria,Jumlah Baris
0,Description kosong,1454
1,Invoice dibatalkan (InvoiceNo diawali C),9288
2,Quantity <= 0,10624
3,Total baris yang dihapus (gabungan),11216


,Metrik,Sebelum,Sesudah,Dihapus
0,Jumlah baris,541909,530693,11216
1,Jumlah invoice unik,25900,20136,5764


Catatan: jumlah pada masing-masing kriteria dapat tumpang tindih, sehingga total gabungan tidak sama dengan penjumlahan sederhana.


Hasil pembersihan:
- Baris berkurang dari **541.909** menjadi **530.693** (11.216 baris dihapus).
- Invoice unik berkurang dari **25.900** menjadi **20.136**.

Selanjutnya, duplikasi baris dengan kombinasi `InvoiceNo` dan `Description` yang sama dihapus. Tujuannya agar satu produk hanya muncul sekali dalam keranjang suatu invoice.

## 3. Persiapan Data untuk Association Rule

Data diubah dari format satu baris per item menjadi format satu baris per invoice. Setelah itu, data ditransformasikan menjadi matriks one-hot: nilai `True` berarti produk dibeli pada invoice tersebut.

In [5]:
# Satu produk dihitung sekali per invoice
basket_df = df_clean.drop_duplicates(subset=["InvoiceNo", "Description"]).copy()

transactions_series = basket_df.groupby("InvoiceNo")["Description"].apply(list)
transactions = transactions_series.tolist()

unique_items = sorted({item for transaction in transactions for item in transaction})
duplicate_lines = len(df_clean) - len(basket_df)

print(f"Baris hasil pembersihan          : {len(df_clean):,}")
print(f"Baris unik invoice-produk        : {len(basket_df):,}")
print(f"Duplikasi invoice-produk dihapus : {duplicate_lines:,}")
print(f"Jumlah transaksi (invoice)       : {len(transactions):,}")
print(f"Jumlah produk unik               : {len(unique_items):,}")
print(f"Rata-rata produk per invoice     : {len(basket_df) / len(transactions):.2f}")

display(transactions_series.head(3).rename("Daftar Produk").to_frame())

Baris hasil pembersihan          : 530,693
Baris unik invoice-produk        : 519,836
Duplikasi invoice-produk dihapus : 10,857
Jumlah transaksi (invoice)       : 20,136
Jumlah produk unik               : 4,057
Rata-rata produk per invoice     : 25.82


,Daftar Produk
InvoiceNo,
536365,"[WHITE HANGING HEART T-LIGHT HOLDER, WHITE METAL LANTERN..."
536366,"[HAND WARMER UNION JACK, HAND WARMER RED POLKA DOT]"
536367,"[ASSORTED COLOUR BIRD ORNAMENT, POPPY'S PLAYHOUSE BEDROO..."


In [6]:
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

basket = pd.DataFrame(te_array, columns=te.columns_)

print(f"Ukuran matriks one-hot: {basket.shape[0]:,} transaksi x {basket.shape[1]:,} produk")
print(f"Total nilai True      : {int(basket.values.sum()):,}")

# Tampilkan lima transaksi pertama pada delapan produk dengan support tertinggi
top_products = basket.sum().sort_values(ascending=False).head(8).index
display(basket.loc[:, top_products].head())

Ukuran matriks one-hot: 20,136 transaksi x 4,057 produk
Total nilai True      : 519,836


,WHITE HANGING HEART T-LIGHT HOLDER,JUMBO BAG RED RETROSPOT,REGENCY CAKESTAND 3 TIER,PARTY BUNTING,LUNCH BAG RED RETROSPOT,ASSORTED COLOUR BIRD ORNAMENT,SET OF 3 CAKE TINS PANTRY DESIGN,PACK OF 72 RETROSPOT CAKE CASES
0,True,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False
2,False,False,False,False,False,True,False,False
3,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False


## 4. Penerapan Algoritma

Algoritma yang digunakan adalah **FP-Growth**. FP-Growth dipilih karena lebih efisien daripada Apriori untuk matriks yang cukup besar: FP-Growth membangun struktur pohon untuk menghindari pemindaian berulang dan pembentukan kandidat yang mahal.

Parameter awal sesuai praktikum:
- `min_support = 0.01` (sekitar 202 invoice dari 20.136 transaksi)
- `min_confidence = 0.5`
- `max_len = 3` agar itemset yang dievaluasi berukuran maksimal tiga produk dan hasilnya lebih mudah ditafsirkan.

In [7]:
min_support_awal = 0.01
min_confidence_awal = 0.5

frequent_itemsets_awal = fpgrowth(
    basket,
    min_support=min_support_awal,
    use_colnames=True,
    max_len=3
)

rules_awal = association_rules(
    frequent_itemsets_awal,
    metric="confidence",
    min_threshold=min_confidence_awal
)

print(f"Jumlah frequent itemset : {len(frequent_itemsets_awal):,}")
print(f"Jumlah aturan awal      : {len(rules_awal):,}")

frequent_itemsets_awal = frequent_itemsets_awal.copy()
frequent_itemsets_awal["jumlah_item"] = frequent_itemsets_awal["itemsets"].apply(len)

display(
    frequent_itemsets_awal
    .groupby("jumlah_item")
    .size()
    .rename("Jumlah Itemset")
    .to_frame()
)

display(
    frequent_itemsets_awal
    .sort_values("support", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

Jumlah frequent itemset : 1,849
Jumlah aturan awal      : 741


,Jumlah Itemset
jumlah_item,
1,804
2,822
3,223


,support,itemsets,jumlah_item
0,0.112237,frozenset({WHITE HANGING HEART T-LIGHT HOLDER}),1
1,0.103894,frozenset({JUMBO BAG RED RETROSPOT}),1
2,0.098778,frozenset({REGENCY CAKESTAND 3 TIER}),1
3,0.083731,frozenset({PARTY BUNTING}),1
4,0.077672,frozenset({LUNCH BAG RED RETROSPOT}),1
5,0.072259,frozenset({ASSORTED COLOUR BIRD ORNAMENT}),1
6,0.068782,frozenset({SET OF 3 CAKE TINS PANTRY DESIGN}),1
7,0.065554,frozenset({PACK OF 72 RETROSPOT CAKE CASES}),1
8,0.063816,frozenset({LUNCH BAG SUKI DESIGN}),1
9,0.063220,frozenset({LUNCH BAG BLACK SKULL.}),1


Dengan parameter awal dihasilkan **741 aturan**. Jumlah ini terlalu banyak untuk dianalisis satu per satu dan banyak aturan berasal dari kombinasi produk niche dengan support yang hanya sedikit di atas 1%. Oleh karena itu, `min_support` dinaikkan dari **0.01 menjadi 0.02** (sekitar 403 invoice), sementara `min_confidence` tetap **0.5**.

Alasannya:
- menaikkan support menyaring pola yang hanya kebetulan muncul pada sangat sedikit invoice;
- mempertahankan confidence 0.5 agar aturan yang sudah cukup kuat tidak dibuang tanpa alasan;
- hasil akhirnya lebih ringkas dan lebih dapat ditindaklanjuti secara bisnis, tanpa menurunkan standar confidence.

In [8]:
min_support = 0.02
min_confidence = 0.5

frequent_itemsets = fpgrowth(
    basket,
    min_support=min_support,
    use_colnames=True,
    max_len=3
)

rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=min_confidence
)

rules["antecedents_text"] = rules["antecedents"].apply(lambda x: " + ".join(sorted(x)))
rules["consequents_text"] = rules["consequents"].apply(lambda x: " + ".join(sorted(x)))

# Support count = perkiraan jumlah invoice yang memuat antecedent dan consequent
rules["support_count"] = (rules["support"] * len(basket)).round().astype(int)

rules_view = rules.sort_values(
    ["lift", "confidence", "support"],
    ascending=False
).copy()

rules_view = rules_view[[
    "antecedents_text", "consequents_text",
    "support_count", "support", "confidence", "lift"
]]
rules_view.columns = [
    "Antecedent (Jika)", "Consequent (Maka)",
    "Jumlah Invoice", "Support", "Confidence", "Lift"
]
rules_view[["Support", "Confidence", "Lift"]] = (
    rules_view[["Support", "Confidence", "Lift"]].round(4)
)

print(f"Jumlah frequent itemset final : {len(frequent_itemsets):,}")
print(f"Jumlah aturan final           : {len(rules):,}")
print(f"Rentang lift                  : {rules['lift'].min():.4f} s.d. {rules['lift'].max():.4f}")
print(f"Semua aturan memiliki lift > 1: {bool((rules['lift'] > 1).all())}")

display(rules_view.reset_index(drop=True))

Jumlah frequent itemset final : 378
Jumlah aturan final           : 61
Rentang lift                  : 5.0768 s.d. 18.5516
Semua aturan memiliki lift > 1: True


,Antecedent (Jika),Consequent (Maka),Jumlah Invoice,Support,Confidence,Lift
0,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER + ROSES REGENCY TEACUP A...,542,0.0269,0.7076,18.5516
1,GREEN REGENCY TEACUP AND SAUCER + ROSES REGENCY TEACUP A...,PINK REGENCY TEACUP AND SAUCER,542,0.0269,0.7057,18.5516
2,PINK REGENCY TEACUP AND SAUCER + ROSES REGENCY TEACUP AN...,GREEN REGENCY TEACUP AND SAUCER,542,0.0269,0.9048,17.9506
3,GREEN REGENCY TEACUP AND SAUCER,PINK REGENCY TEACUP AND SAUCER + ROSES REGENCY TEACUP AN...,542,0.0269,0.5340,17.9506
4,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,633,0.0314,0.8264,16.3939
5,GREEN REGENCY TEACUP AND SAUCER,PINK REGENCY TEACUP AND SAUCER,633,0.0314,0.6236,16.3939
6,GREEN REGENCY TEACUP AND SAUCER + PINK REGENCY TEACUP AN...,ROSES REGENCY TEACUP AND SAUCER,542,0.0269,0.8562,16.1738
7,ROSES REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER + PINK REGENCY TEACUP AN...,542,0.0269,0.5084,16.1738
8,GARDENERS KNEELING PAD KEEP CALM,GARDENERS KNEELING PAD CUP OF TEA,546,0.0271,0.5980,15.8864
9,GARDENERS KNEELING PAD CUP OF TEA,GARDENERS KNEELING PAD KEEP CALM,546,0.0271,0.7203,15.8864


## 5. Evaluasi Aturan Asosiasi

Tabel di atas menampilkan seluruh aturan final, diurutkan berdasarkan `lift`. Kolom **Jumlah Invoice** adalah *support count*, yaitu banyaknya invoice yang memuat seluruh item dalam aturan (antecedent dan consequent), bukan jumlah invoice yang memuat antecedent saja. Tiga aturan dengan lift tertinggi ditampilkan kembali di bawah ini untuk interpretasi.

In [9]:
top3 = rules_view.sort_values(
    ["Lift", "Confidence", "Support"],
    ascending=False
).head(3).reset_index(drop=True)

display(top3)

,Antecedent (Jika),Consequent (Maka),Jumlah Invoice,Support,Confidence,Lift
0,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER + ROSES REGENCY TEACUP A...,542,0.0269,0.7076,18.5516
1,GREEN REGENCY TEACUP AND SAUCER + ROSES REGENCY TEACUP A...,PINK REGENCY TEACUP AND SAUCER,542,0.0269,0.7057,18.5516
2,PINK REGENCY TEACUP AND SAUCER + ROSES REGENCY TEACUP AN...,GREEN REGENCY TEACUP AND SAUCER,542,0.0269,0.9048,17.9506


### Interpretasi tiga aturan terbaik

Ketiga aturan teratas berasal dari kombinasi tiga varian **Regency teacup and saucer**. Artinya, ketiganya bukan tiga temuan yang sepenuhnya independen, melainkan tiga sudut pandang dari satu pola pembelian yang sangat kuat.

1. **Pink Regency Teacup and Saucer → Green + Roses Regency Teacup and Saucer**
   - Support **2,69%** (542 invoice), confidence **70,76%**, lift **18,55**.
   - Pelanggan yang membeli varian *pink* memiliki kecenderungan 18,55 kali lebih tinggi untuk juga membeli varian *green* dan *roses* dibandingkan pelanggan acak.
   - **Implikasi bisnis:** tampilkan ketiga varian sebagai satu koleksi dan tawarkan bundling “lengkapi koleksi cangkir teh”. Rekomendasi ini cocok untuk keranjang belanja online maupun penempatan produk yang berdekatan di toko.

2. **Green + Roses Regency Teacup and Saucer → Pink Regency Teacup and Saucer**
   - Support **2,69%** (542 invoice), confidence **70,57%**, lift **18,55**.
   - Arah sebaliknya juga sangat kuat. Ini menunjukkan hubungan timbal balik: pembeli dua varian lain juga sangat mungkin membeli varian *pink*.
   - **Implikasi bisnis:** jangan hanya mengandalkan satu arah cross-selling. Ketiga varian perlu tersedia bersama, misalnya dalam satu set hadiah atau satu area display.

3. **Pink + Roses Regency Teacup and Saucer → Green Regency Teacup and Saucer**
   - Support **2,69%** (542 invoice), confidence **90,48%**, lift **17,95**.
   - Jika pelanggan sudah mengambil varian *pink* dan *roses*, sekitar 90% di antaranya juga mengambil varian *green*. Varian *green* berperan sebagai produk pelengkap yang kuat.
   - **Implikasi bisnis:** jadikan varian *green* sebagai rekomendasi utama ketika dua varian lainnya sudah ada di keranjang, dan pastikan stoknya cukup agar peluang upsell tidak hilang.

Secara keseluruhan, pola ini menunjukkan bahwa pelanggan cenderung membeli **beberapa varian warna/desain dari satu keluarga produk**, bukan hanya satu unit. Strategi yang tepat adalah menjualnya sebagai koleksi, bukan sebagai produk yang terpisah-pisah.

### Pola lain yang menarik

Selain teacup, beberapa aturan dengan lift tinggi menunjukkan pola yang dapat dimanfaatkan:

| Aturan | Support | Confidence | Lift | Makna singkat |
|---|---:|---:|---:|---|
| Gardeners Kneeling Pad Keep Calm → Cup of Tea | 2,71% | 59,80% | 15,89 | Dua desain kneeling pad dibeli sebagai pilihan/pasangan. |
| Spaceboy Lunch Box ↔ Dolly Girl Lunch Box | 2,67% | 59,78% / 63,29% | 14,16 | Kotak makan anak dengan tema berbeda dibeli bersama, misalnya untuk dua anak. |
| Jumbo Bag Pears → Jumbo Bag Apples | 2,02% | 68,24% | 14,05 | Pembeli memilih lebih dari satu desain tas belanja. |

Pola-pola ini dapat diuji melalui bundling, rekomendasi “produk serupa”, atau penempatan display koleksi.

## Kesimpulan

1. Dataset awal memiliki **541.909 baris** dan **25.900 invoice**. Setelah pembersihan, tersisa **530.693 baris** dan **20.136 invoice** yang dianalisis.
2. Matriks one-hot yang terbentuk berukuran **20.136 transaksi × 4.057 produk**.
3. Dengan parameter awal `min_support = 0.01` dan `min_confidence = 0.5` dihasilkan **741 aturan**, yang terlalu banyak untuk dianalisis secara praktis.
4. Setelah `min_support` dinaikkan menjadi **0.02** dan confidence dipertahankan **0.5**, diperoleh **61 aturan** dengan lift minimum di atas 1.
5. Tiga aturan dengan lift tertinggi menunjukkan asosiasi sangat kuat antarvarian **Regency teacup and saucer**. Rekomendasi utamanya adalah menjual ketiga varian sebagai satu koleksi dan memakai varian *green* sebagai pelengkap ketika pelanggan membeli dua varian lainnya.

**Keterbatasan:** association rule menunjukkan hubungan antaritem, bukan sebab-akibat. Aturan juga dibatasi pada itemset maksimal tiga produk dan hanya mencakup pola dengan support minimal 2% dari seluruh invoice, sehingga pola yang sangat jarang tidak akan muncul.